In [1]:
!pkill -f python
!pkill -f ngrok
print("Port 5000 and background processes cleared.")

'pkill' is not recognized as an internal or external command,
operable program or batch file.


Port 5000 and background processes cleared.


'pkill' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
# Install required libraries
!pip install flask openai pyngrok nest-asyncio

# Configure your ngrok auth token securely using Colab secrets
from google.colab import userdata
import os

try:
    # Retrieve your token from the Secrets manager (🔑 in left sidebar)
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    !ngrok authtoken {NGROK_AUTHTOKEN}
    print("ngrok authtoken configured successfully!")
except Exception as e:
    print("Error: Make sure you have added your NGROK_AUTHTOKEN to the Colab Secrets (🔑) and enabled notebook access.")

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------ --------------------- 1.0/2.3 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 6.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------------------------------- - 2.1/2.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.0 MB/s eta 0:00:00

   ----------------------------------------  0/23 [typing-extensions]
   - --------------------------------------  1/23 [truststore]
   --- ------------------------------------  2/23 [sniffio]
   ----- ----------------------------------  3/23 [PyYAML]
   ----- ----------------------------------  3/23 [PyYAML]
   ----- ----------------------------------  3/23 [PyYAML]
   ----- ----------------------------------  3/23 [PyYAML]
   ----- ----------------------------------  3/23


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google'

In [ ]:
import os

# Create templates directory
os.makedirs("templates", exist_ok=True)
print("Directory 'templates' ready.")

Directory 'templates' ready.


In [ ]:
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Emergency Dispatch AI Assistant</title>
    <style>
        body { font-family: Arial, sans-serif; background-color: #f4f4f9; margin: 0; padding: 0; display: flex; justify-content: center; align-items: center; height: 100vh; }
        .chat-container { width: 400px; background: white; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-direction: column; overflow: hidden; height: 600px; }
        .chat-header { background: #d9534f; color: white; padding: 15px; text-align: center; font-weight: bold; font-size: 18px; }
        .chat-messages { flex: 1; padding: 15px; overflow-y: auto; display: flex; flex-direction: column; gap: 10px; background: #fafafa; }
        .message { padding: 10px 14px; border-radius: 6px; max-width: 80%; line-height: 1.4; word-wrap: break-word; }
        .message.user { background: #0275d8; color: white; align-self: flex-end; }
        .message.bot { background: #e9ecef; color: #333; align-self: flex-start; }
        .chat-input-area { display: flex; border-top: 1px solid #ddd; padding: 10px; background: white; }
        .chat-input-area input { flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 4px; outline: none; }
        .chat-input-area button { background: #d9534f; color: white; border: none; padding: 10px 15px; margin-left: 8px; border-radius: 4px; cursor: pointer; font-weight: bold; }
        .chat-input-area button:hover { background: #c9302c; }
    </style>
</head>
<body>

<div class="chat-container">
    <div class="chat-header">EMERGENCY DISPATCH INTAKE</div>
    <div id="chatMessages" class="chat-messages">
        <div class="message bot">911/Emergency AI Intake active. What is your emergency? (Medical, Fire, or Police)</div>
    </div>
    <div class="chat-input-area">
        <input type="text" id="userInput" placeholder="Type your situation here..." autofocus>
        <button onclick="sendMessage()">Send</button>
    </div>
</div>

<script>
    const chatHistory = [];
    document.getElementById("userInput").addEventListener("keypress", function(event) {
        if (event.key === "Enter") { sendMessage(); }
    });

    async function sendMessage() {
        const inputField = document.getElementById("userInput");
        const messageText = inputField.value.trim();
        if (!messageText) return;

        appendMessage(messageText, "user");
        chatHistory.push({ role: "user", content: messageText });
        inputField.value = "";

        const chatMessages = document.getElementById("chatMessages");
        const loadingId = "loading-" + Date.now();
        const loadingDiv = document.createElement("div");
        loadingDiv.className = "message bot";
        loadingDiv.id = loadingId;
        loadingDiv.innerText = "Analyzing and routing...";
        chatMessages.appendChild(loadingDiv);
        chatMessages.scrollTop = chatMessages.scrollHeight;

        try {
            const response = await fetch("/chat", {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify({ message: messageText, history: chatHistory })
            });
            const data = await response.json();
            document.getElementById(loadingId).remove();

            if (data.reply) {
                appendMessage(data.reply, "bot");
                chatHistory.push({ role: "assistant", content: data.reply });
            } else {
                appendMessage("Error receiving dispatch response.", "bot");
            }
        } catch (error) {
            document.getElementById(loadingId).remove();
            appendMessage("Network connection error. Call emergency services directly.", "bot");
        }
    }

    function appendMessage(text, sender) {
        const chatMessages = document.getElementById("chatMessages");
        const msgDiv = document.createElement("div");
        msgDiv.className = `message ${sender}`;
        msgDiv.innerText = text;
        chatMessages.appendChild(msgDiv);
        chatMessages.scrollTop = chatMessages.scrollHeight;
    }
</script>

</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_content)
print("templates/index.html successfully created.")

templates/index.html successfully created.


In [ ]:
import threading
import socket
from flask import Flask, render_template, request, jsonify
from openai import OpenAI
from pyngrok import ngrok
import nest_asyncio
from google.colab import userdata

# Apply nest_asyncio to let event loops run concurrently in Colab
nest_asyncio.apply()

app = Flask(__name__)

# Retrieve your OpenAI API Key securely from Colab Secrets
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=openai_key)
except Exception:
    print("Warning: 'OPENAI_API_KEY' not found in Colab Secrets. Falling back to placeholder.")
    client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

SYSTEM_PROMPT = """
You are an AI Emergency Dispatch Intake Assistant. Your job is to calmly, clearly, and quickly gather critical information from a caller experiencing an emergency.
You must collect 4 key pieces of information sequentially or as naturally as possible:
1. Exact Location (Address or landmarks)
2. Nature of Emergency (Medical, Fire, Police, Accident)
3. Immediate Danger / Safety Status (Is anyone else in danger?)
4. Specific details (Injuries, consciousness, suspect description, etc.)

Tone requirements:
- Calm, authoritative, empathetic, and extremely concise.
- Do not write long paragraphs. Keep responses under 2-3 sentences.
- Always reassure them that emergency services are being notified based on their inputs.
"""

@app.route("/")
def index():
    return render_template("index.html")

@app.route("/chat", methods=["POST"])
def chat():
    user_message = request.json.get("message", "")
    chat_history = request.json.get("history", [])

    if not user_message:
        return jsonify({"error": "No message provided"}), 400

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for msg in chat_history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": user_message})

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.3,
            max_tokens=150
        )
        bot_reply = response.choices[0].message.content
        return jsonify({"reply": bot_reply})
    except Exception as e:
        return jsonify({"reply": "System Error: Dispatch connection unstable. Please dial your local emergency number directly."}), 500

def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

port = 5000
if is_port_in_use(port):
    print(f"Port {port} is already active. Re-using the background instance or connecting ngrok to it.")
else:
    def run_flask():
        try:
            app.run(port=port, debug=False, use_reloader=False)
        except Exception as e:
            print(f"Flask failed to start: {e}")

    # Start Flask in a background thread so Colab doesn't freeze
    t = threading.Thread(target=run_flask, daemon=True)
    t.start()

try:
    # Open an ngrok tunnel to the active port
    public_url = ngrok.connect(port)
    print(f"\n * PUBLIC CHATBOT URL: {public_url} \nClick the link above to test your Emergency Dispatch Bot!")
except Exception as e:
    print(f"\nNgrok Tunnel Connection Failed: {e}")
    print("Please verify that NGROK_AUTHTOKEN is configured correctly in the first cell and executed.")

Port 5000 is already active. Re-using the background instance or connecting ngrok to it.


ERROR:pyngrok.process.ngrok:t=2026-10-17T04:46:50+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-17T04:46:50+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"



Ngrok Tunnel Connection Failed: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.
Please verify that NGROK_AUTHTOKEN is configured correctly in the first cell and executed.


In [ ]:
import os
import threading
from flask import Flask, render_template, request, jsonify
from openai import OpenAI
from pyngrok import ngrok, conf
import nest_asyncio

# ==============================================================================
# PASTE YOUR API KEYS DIRECTLY HERE:
# ==============================================================================
OPENAI_API_KEY = "sk-proj-YOUR_ACTUAL_OPENAI_KEY_HERE"
NGROK_AUTHTOKEN = "YOUR_ACTUAL_NGROK_AUTHTOKEN_HERE"
# ==============================================================================

# Configure ngrok authentication
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 1. Setup Flask Template Structure
os.makedirs("templates", exist_ok=True)

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Emergency Dispatch AI Assistant</title>
    <style>
        body { font-family: Arial, sans-serif; background-color: #f4f4f9; margin: 0; padding: 0; display: flex; justify-content: center; align-items: center; height: 100vh; }
        .chat-container { width: 400px; background: white; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-direction: column; overflow: hidden; height: 600px; }
        .chat-header { background: #d9534f; color: white; padding: 15px; text-align: center; font-weight: bold; font-size: 18px; }
        .chat-messages { flex: 1; padding: 15px; overflow-y: auto; display: flex; flex-direction: column; gap: 10px; background: #fafafa; }
        .message { padding: 10px 14px; border-radius: 6px; max-width: 80%; line-height: 1.4; word-wrap: break-word; }
        .message.user { background: #0275d8; color: white; align-self: flex-end; }
        .message.bot { background: #e9ecef; color: #333; align-self: flex-start; }
        .chat-input-area { display: flex; border-top: 1px solid #ddd; padding: 10px; background: white; }
        .chat-input-area input { flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 4px; outline: none; }
        .chat-input-area button { background: #d9534f; color: white; border: none; padding: 10px 15px; margin-left: 8px; border-radius: 4px; cursor: pointer; font-weight: bold; }
        .chat-input-area button:hover { background: #c9302c; }
    </style>
</head>
<body>
<div class="chat-container">
    <div class="chat-header">EMERGENCY DISPATCH INTAKE</div>
    <div id="chatMessages" class="chat-messages">
        <div class="message bot">911/Emergency AI Intake active. What is your emergency? (Medical, Fire, or Police)</div>
    </div>
    <div class="chat-input-area">
        <input type="text" id="userInput" placeholder="Type your situation here..." autofocus>
        <button onclick="sendMessage()">Send</button>
    </div>
</div>
<script>
    const chatHistory = [];
    document.getElementById("userInput").addEventListener("keypress", function(event) {
        if (event.key === "Enter") { sendMessage(); }
    });
    async function sendMessage() {
        const inputField = document.getElementById("userInput");
        const messageText = inputField.value.trim();
        if (!messageText) return;
        appendMessage(messageText, "user");
        chatHistory.push({ role: "user", content: messageText });
        inputField.value = "";
        const chatMessages = document.getElementById("chatMessages");
        const loadingId = "loading-" + Date.now();
        const loadingDiv = document.createElement("div");
        loadingDiv.className = "message bot";
        loadingDiv.id = loadingId;
        loadingDiv.innerText = "Analyzing and routing...";
        chatMessages.appendChild(loadingDiv);
        chatMessages.scrollTop = chatMessages.scrollHeight;
        try {
            const response = await fetch("/chat", {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify({ message: messageText, history: chatHistory })
            });
            const data = await response.json();
            document.getElementById(loadingId).remove();
            if (data.reply) {
                appendMessage(data.reply, "bot");
                chatHistory.push({ role: "assistant", content: data.reply });
            } else {
                appendMessage("Error receiving dispatch response.", "bot");
            }
        } catch (error) {
            document.getElementById(loadingId).remove();
            appendMessage("Network connection error. Call emergency services directly.", "bot");
        }
    }
    function appendMessage(text, sender) {
        const chatMessages = document.getElementById("chatMessages");
        const msgDiv = document.createElement("div");
        msgDiv.className = `message ${sender}`;
        msgDiv.innerText = text;
        chatMessages.appendChild(msgDiv);
        chatMessages.scrollTop = chatMessages.scrollHeight;
    }
</script>
</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_content)

# 2. Setup Flask App & OpenAI
nest_asyncio.apply()
app = Flask(__name__)
client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """
You are an AI Emergency Dispatch Intake Assistant. Your job is to calmly, clearly, and quickly gather critical information from a caller experiencing an emergency.
You must collect 4 key pieces of information sequentially or as naturally as possible:
1. Exact Location (Address or landmarks)
2. Nature of Emergency (Medical, Fire, Police, Accident)
3. Immediate Danger / Safety Status (Is anyone else in danger?)
4. Specific details (Injuries, consciousness, suspect description, etc.)

Tone requirements:
- Calm, authoritative, empathetic, and extremely concise.
- Do not write long paragraphs. Keep responses under 2-3 sentences.
- Always reassure them that emergency services are being notified based on their inputs.
"""

@app.route("/")
def index():
    return render_template("index.html")

@app.route("/chat", methods=["POST"])
def chat():
    user_message = request.json.get("message", "")
    chat_history = request.json.get("history", [])
    if not user_message:
        return jsonify({"error": "No message provided"}), 400

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for msg in chat_history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": user_message})

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.3,
            max_tokens=150
        )
        return jsonify({"reply": response.choices[0].message.content})
    except Exception as e:
        return jsonify({"reply": "System Error: Dispatch connection unstable. Please dial your local emergency number directly."}), 500

# Kill any existing old background tunnels first to free port 5000
!pkill -f python
!pkill -f ngrok

def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

# Start Flask & Ngrok
t = threading.Thread(target=run_flask)
t.start()

public_url = ngrok.connect(5000)
print(f"\n🚀 PUBLIC CHATBOT URL: {public_url} \nClick the link above to test your Emergency Dispatch Bot!")